# 06 - Reranking
## Shakespeare Evidence Desk

- **Goal:** reorder a *fixed* set of retrieved candidates so the most relevant passages come first — without searching the corpus again.
- **Experiment axis:** no reranking vs. bi-encoder semantic reranking vs. bounded listwise LLM reranking (the candidate set is held fixed; only the reordering changes).
- **Loads from Step 05:** `.artifacts/retrieval-handoff.json` (the selected retrieval method) and `.artifacts/index.json` (the rows + vectors), replayed to rebuild each question's candidate pool.
- **Exports:** `.artifacts/reranking-handoff.json` — the reranking decision for stage 07 (context building).

## Why This Stage

*In general — what reranking is and why a RAG system reaches for it.* Retrieval (stage 05) is built for **recall**: it scans the whole corpus cheaply and returns a broad pool of candidates, accepting that the single best passage may not be ranked first. Reranking is the opposite trade — it ignores the corpus entirely and spends a **more expensive, more precise** model on the small pool retrieval already returned, to pull the truly best evidence to the very top. It buys **precision at the top** (Recall@1, nDCG) with latency and cost, on a fixed, bounded set.

The core decision reranking owns is *whether a second, stronger relevance judgment is worth paying for* — and if so, which one: a cheap bi-encoder re-score, a bounded LLM that reads the passages and orders them, or nothing at all. Because a reranker only helps if it beats retrieval's own order by enough to justify its cost, this stage is measured, not assumed: we keep a reranker **only** when the quality gain clears a margin.

## Conceptual Foundations

### Retrieval casts a wide net; reranking sharpens the top

Stage 05 optimized for **recall** — get a relevant passage *somewhere* into a broad candidate pool, cheaply, over the whole corpus. But the generator (stage 08) reads only the first few passages, so *where* in the pool the best evidence sits matters enormously. **Reranking** reorders that fixed pool with a stronger relevance model so the best passage lands at position 1, not position 9. It never searches the corpus and never adds new passages: **a passage retrieval missed can never be reranked in.** Reranking raises precision at the top; it cannot raise recall.

### Two-stage retrieve-then-rerank, and why the split exists

The standard pattern is **retrieve-then-rerank**: a fast, cheap retriever (bi-encoder + ANN, BM25) narrows millions of passages to tens, then a slow, accurate reranker re-scores just those tens. The split exists because the accurate model is too expensive to run over the whole corpus — a cross-encoder that reads *(query, passage)* jointly would have to score every document — but perfectly affordable on a pool of 20. Retrieval picks the candidates; reranking picks the order.

### Three ways to rerank (cheap → expensive)

- **None (control).** Keep retrieval's own order. The honest baseline every reranker must beat — if reordering doesn't help, you pay latency for nothing.
- **Bi-encoder semantic.** Re-score each candidate by the cosine similarity between the query embedding and the passage embedding. Cheap (vectors already exist, no new model) but uses the *same* signal a dense retriever already used — so if the pool came from dense search, it reproduces that order. A transparent stand-in for a true cross-encoder.
- **Cross-encoder / LLM.** Feed the query **and** the passage text to one model that judges them *together*. This is strictly more powerful than a bi-encoder: it can see that a passage mentions the exact thing asked, not just that it is topically near. The cost is a model call per pool (or per passage). We run a **bounded listwise LLM** reranker: one capped call orders the whole pool at once.

### Pointwise vs. listwise LLM reranking

An LLM reranker can score candidates **pointwise** (one call per passage → a relevance score each) or **listwise** (one call sees the whole numbered pool and returns an ordering). Listwise is far cheaper (one call per query, not one per candidate) and lets the model compare passages against each other; its risk is instability if the list is long or the model drops items. We bound it — a small pool, truncated snippets, one call — and defensively re-append anything the model omits.

### The knobs reranking owns

- **pool size** — how many retrieved candidates to rerank. Larger pool = more chances to recover a buried relevant passage, but more to read (LLM cost) and more noise. It is a recall-feeding-precision dial, distinct from the final top-k.
- **adopt-or-not margin** — the minimum quality gain over *no reranking* required to pay for a reranker at all. Reranking is optional; this margin encodes that.
- **boundedness (for the LLM arm)** — caps on pool size, snippet length, and calls per query that hold latency and cost predictable.

### What reranking is *not* (clean stage boundaries)

- **Not retrieval (05):** it does not search the corpus or change recall; it only reorders a fixed set.
- **Not query rewriting (09):** it takes the query as written.
- **Not context building (07):** it orders passages; it does not dedupe, fetch parents, or fit a token budget.
- **Not generation (08):** it supplies ordered evidence; it writes no answer.

### Common misconceptions and failure modes

- **"A reranker always helps."** Only when retrieval's order was suboptimal *and* the reranker's judgment is better. On a small, clean pool where retrieval already nailed the top, reranking can tie — or hurt, by promoting a fluent-but-wrong passage. Measure before adopting.
- **"The LLM reranker is the obvious winner."** It is the most powerful judge, but the most expensive and the slowest, and it can be swayed by surface fluency. It wins only when its gain clears the cost margin.
- **"Rerank a huge pool to be safe."** Bigger pools cost more to read and dilute the signal; past a point they add noise, not recall.
- **"Reranking fixes bad retrieval."** It cannot. If the relevant passage is not in the pool, no reordering recovers it — fix retrieval (05) or chunking (02) instead.

## 1. Load the Previous Selection

- **Purpose:** start from stage 05's exported decision and stage 04's index, not hidden kernel state. Reranking only *reorders* what retrieval returns, so the retrieval method, top-k, index, and metadata schema are fixed input here.
- **What the code does:** locates the repository root, reads `retrieval-handoff.json` (the selected method and its parameters) and `index.json` (the rows with vectors + provenance, and the IVF structure), reconstructs exactly the stage-05 **selected retrieval method** (BM25, dense, or hybrid/RRF) so it can be *replayed* to build each question's candidate pool, and rebuilds the **answer key** from the labeled questions (reusing the citation parser from notebooks 02–05) so Section 4 can grade the reranked order.
- **Fails loudly, early:** if `index.json` or `retrieval-handoff.json` is absent, run notebooks 04/05 first. If the index was built on offline-fallback vectors (`embedding_backend != 'hosted'`), the notebook **stops here** — reranking fake candidates is meaningless.
- **Expected result:** the index type and metric, the fixed upstream retrieval configuration (method, top-k, RRF constant), and how many labeled questions resolved to an indexed scene the grader can use. No network call happens here.

In [ ]:
import json
import math
import re
from collections import defaultdict
from pathlib import Path

# Work from the repository root so the notebook and the package read the same .env and artifacts.
ROOT = Path.cwd().resolve()
if not (ROOT / '.env.example').is_file() and (ROOT.parent / '.env.example').is_file():
    ROOT = ROOT.parent
if not (ROOT / '.env.example').is_file():
    raise RuntimeError('Use the repository as the notebook working directory.')

ARTIFACTS = ROOT / '.artifacts'
index_path = ARTIFACTS / 'index.json'
retrieval_handoff_path = ARTIFACTS / 'retrieval-handoff.json'
questions_path = ARTIFACTS / 'eval-questions.json'
if not index_path.is_file():
    raise RuntimeError('Missing .artifacts/index.json. Run notebook 04 first.')
if not retrieval_handoff_path.is_file():
    raise RuntimeError('Missing .artifacts/retrieval-handoff.json. Run notebook 05 first.')

index_payload = json.loads(index_path.read_text(encoding='utf-8'))
retrieval_handoff = json.loads(retrieval_handoff_path.read_text(encoding='utf-8'))
evaluation_questions = json.loads(questions_path.read_text(encoding='utf-8')) if questions_path.is_file() else {'development': [], 'heldout': []}

index_type = index_payload['index_type']          # 'ivf' or 'exact', the stage-04 selection
metric = index_payload['metric']                  # 'inner_product' (unit vectors, so dot == cosine)
stored_dims = index_payload['dims']
embedding_backend = index_payload.get('embedding_backend', 'unknown')
records = index_payload['records']                # chunk_id, work_id, work, scenes, speakers, text, embedding

# Stop now if the upstream vectors are not real: reranking offline-fallback candidates is meaningless.
if embedding_backend != 'hosted':
    raise RuntimeError(
        f"index.json was built on the '{embedding_backend}' backend, not hosted Azure. "
        "Re-run notebooks 03-05 against Azure before reranking.")

# ---- the fixed retrieval configuration this stage reorders (never re-decided here) ----
SELECTED_METHOD = retrieval_handoff['selected_method']       # bm25 | dense | hybrid-rrf
TOP_K = retrieval_handoff['top_k']                           # final candidates graded after reranking
RRF_K = retrieval_handoff.get('rrf_k') or 60
bm25_params = retrieval_handoff.get('bm25_params', {'k1': 1.5, 'b': 0.75})

vectors = [r['embedding'] for r in records]
N = len(vectors)
work_titles = {r['work_id']: r['work'] for r in records}


def dot(a, b):
    # Score between two unit vectors; equals cosine similarity. Higher = more relevant.
    return sum(x * y for x, y in zip(a, b))


def l2_normalize(vector):
    length = math.sqrt(sum(value * value for value in vector))
    return [value / length for value in vector] if length else vector


# ---- reconstruct stage 04's dense search exactly (reranking queries the same index) ----
def exact_search_rows(query_vector, candidate_rows=None):
    pool = range(N) if candidate_rows is None else candidate_rows
    return sorted(pool, key=lambda i: dot(query_vector, vectors[i]), reverse=True)


if index_type == 'ivf':
    ivf = index_payload['ivf']
    ivf_centroids = ivf['centroids']
    ivf_lists = {int(cluster): rows for cluster, rows in ivf['lists'].items()}
    nprobe = index_payload['params']['nprobe']

    def dense_search(query_vector, k, candidate_rows=None):
        # Open only the nprobe nearest clusters (the stage-04 dial), then exact-score within that pool.
        nearest = sorted(range(len(ivf_centroids)),
                         key=lambda c: dot(query_vector, ivf_centroids[c]), reverse=True)[:nprobe]
        pool = [i for c in nearest for i in ivf_lists.get(c, ())]
        if candidate_rows is not None:
            allowed = set(candidate_rows)
            pool = [i for i in pool if i in allowed]
        return exact_search_rows(query_vector, pool)[:k]
else:
    nprobe = None

    def dense_search(query_vector, k, candidate_rows=None):
        return exact_search_rows(query_vector, candidate_rows)[:k]


# ---- rebuild the keyword (BM25) and hybrid (RRF) arms so whichever method stage 05 chose can be replayed ----
_WORD_PATTERN = re.compile(r"[a-z0-9']+")


def tokenize(text):
    return _WORD_PATTERN.findall(text.lower())


class BM25:
    """The same transparent Okapi BM25 keyword index built in notebook 05, reused to replay its pool."""

    def __init__(self, documents, term_saturation=1.5, length_strength=0.75):
        self.k1 = term_saturation
        self.b = length_strength
        self.tokenized_docs = [tokenize(text) for text in documents]
        self.doc_lengths = [len(tokens) for tokens in self.tokenized_docs]
        self.avg_len = (sum(self.doc_lengths) / len(self.tokenized_docs)) if self.tokenized_docs else 0.0
        self.term_counts_per_doc = []
        document_frequency = defaultdict(int)
        for tokens in self.tokenized_docs:
            term_counts = defaultdict(int)
            for term in tokens:
                term_counts[term] += 1
            self.term_counts_per_doc.append(term_counts)
            for term in term_counts:
                document_frequency[term] += 1
        total_documents = len(self.tokenized_docs)
        self.idf = {
            term: math.log(1 + (total_documents - doc_count + 0.5) / (doc_count + 0.5))
            for term, doc_count in document_frequency.items()
        }

    def search(self, query, top_k, candidate_rows=None):
        query_terms = [term for term in tokenize(query) if term in self.idf]
        candidate_indices = range(len(self.tokenized_docs)) if candidate_rows is None else candidate_rows
        scored_rows = []
        for row_index in candidate_indices:
            term_counts = self.term_counts_per_doc[row_index]
            doc_length = self.doc_lengths[row_index]
            score = 0.0
            for term in query_terms:
                term_count = term_counts.get(term, 0)
                if not term_count:
                    continue
                length_normalizer = self.k1 * (1 - self.b + self.b * doc_length / (self.avg_len or 1))
                score += self.idf[term] * (term_count * (self.k1 + 1)) / (term_count + length_normalizer)
            if score > 0:
                scored_rows.append((row_index, score))
        scored_rows.sort(key=lambda row_and_score: row_and_score[1], reverse=True)
        return [row_index for row_index, _ in scored_rows[:top_k]]


bm25 = BM25([record['text'] for record in records],
            term_saturation=bm25_params.get('k1', 1.5), length_strength=bm25_params.get('b', 0.75))


def reciprocal_rank_fusion(rankings, damping, top_k):
    fused_scores = defaultdict(float)
    for ranking in rankings:
        for rank, row_index in enumerate(ranking):
            fused_scores[row_index] += 1.0 / (damping + rank + 1)
    ranked_rows = sorted(fused_scores.items(), key=lambda row_and_score: row_and_score[1], reverse=True)
    return [row_index for row_index, _ in ranked_rows[:top_k]]


def retrieve_pool(query_text, query_vector, pool_size):
    """Replay the stage-05 selected method to produce the FIXED candidate pool the rerankers reorder."""
    if SELECTED_METHOD == 'bm25':
        return bm25.search(query_text, pool_size)
    if SELECTED_METHOD == 'dense':
        return dense_search(query_vector, pool_size)
    dense_ranking = dense_search(query_vector, pool_size)
    keyword_ranking = bm25.search(query_text, pool_size)
    return reciprocal_rank_fusion([dense_ranking, keyword_ranking], RRF_K, pool_size)


# ---- answer key: turn each labeled question into the set of index rows holding its cited scene ----
ROMAN_VALUES = {'I': 1, 'V': 5, 'X': 10, 'L': 50, 'C': 100, 'D': 500, 'M': 1000}


def roman_to_int(value):
    if value is None:
        return None
    text = str(value).strip().upper()
    if text.isdigit():
        return int(text)
    total, previous = 0, 0
    for char in reversed(text):
        if char not in ROMAN_VALUES:
            return None
        current = ROMAN_VALUES[char]
        total += -current if current < previous else current
        previous = max(previous, current)
    return total or None


lowercase_title = {work_id: title.lower() for work_id, title in work_titles.items()}


def cited_locations(question):
    # Read the label's citation -> list of (work, act, scene); uses the label only, never the question's words.
    label_text = (question.get('question', '') + ' ' + question.get('expected_evidence', '')).lower()
    works = [work_id for work_id, title in lowercase_title.items() if title in label_text]
    act_scene_pairs = re.findall(r'act\s+(\d+|[ivxlc]+)\s*,?\s*scene\s+(\d+|[ivxlc]+)',
                                 question.get('expected_evidence', '').lower())
    acts_scenes = [(roman_to_int(act), roman_to_int(scene)) for act, scene in act_scene_pairs]
    return [(work_id, act, scene) for work_id in works for act, scene in acts_scenes]


def row_in_scene(record, work_id, act, scene):
    return record['work_id'] == work_id and any(a == act and s == scene for a, s in record['scenes'])


# For each answerable question: the relevant row set (graded truth) used to score the reranked order.
answer_key = []
for question in evaluation_questions.get('development', []):
    if not question.get('answerable'):
        continue
    locations = [(w, a, s) for w, a, s in cited_locations(question) if a is not None and s is not None]
    relevant_rows = {i for i, record in enumerate(records)
                     for (w, a, s) in locations if row_in_scene(record, w, a, s)}
    if relevant_rows:
        answer_key.append((question, relevant_rows))

print(f'Loaded {N:,} indexed rows ({index_type}, metric={metric}, {stored_dims} dims) from the {embedding_backend} backend.')
print(f'Fixed upstream (stage 05): method={SELECTED_METHOD}, top-k={TOP_K}, RRF k={RRF_K}.')
print(f'Answer key: {len(answer_key)} labeled dev question(s) resolved to an indexed scene.')

## 2. Explain the Alternatives

A reranker is **good** when it lifts a passage from the cited scene to the very top of the pool — raising precision at rank 1 and nDCG — by *enough* to justify the latency and cost it adds. We **fix everything upstream** (the retrieved candidate pool, the top-k, the index) and vary only the *reordering strategy*.

### What we actually compare vs. what we only explain

| Method | Role here | Why |
|---|---|---|
| **No reranking** | ✅ **compared** (control) | retrieval's own order is the baseline every reranker must beat |
| **Bi-encoder semantic** | ✅ **compared** | cheap re-score by query↔passage cosine; the standard lightweight reranker |
| **Bounded listwise LLM** | ✅ **compared** | a strong cross-encoder-style judge reading query + passages together |
| **Cross-encoder (trained, e.g. MiniLM/BGE reranker)** | ○ **explained, not run** | the production-grade reranker; needs a model we don't host locally — the LLM arm stands in for it, and Azure AI Search's semantic ranker is the production target |
| **Pointwise LLM scoring** | ○ **explained, not run** | one call per candidate — strictly costlier than the listwise arm for the same pool |
| **ColBERT late-interaction** | ○ **explained, not run** | token-level reranking; needs a different index than stage 04 built |

So three arms are compared head-to-head; the rest are explained so the choice makes sense.

### How each arm actually works (a line or two)

- **No reranking** — return the pool in retrieval's order. Zero added latency or cost; the yardstick for whether reordering earns its keep.
- **Bi-encoder semantic** — score each candidate by the cosine between the query vector and the passage vector (both already computed), highest first. It re-applies the *dense* signal to the whole pool, so it can rescue a semantically-strong passage that BM25 or hybrid ranked low — but it adds nothing new when the pool already came from dense search.
- **Bounded listwise LLM** — send the query and the numbered pool (passages truncated) to the chat model in **one** call and ask for a relevance ordering. The model reads query and passages *together*, so it can judge actual answerhood, not just vector nearness — at the cost of a model call and tokens. "Bounded" caps the pool, the snippet length, and the calls so latency and cost stay predictable.

### Why this is the experiment we run

Reranking's whole premise is paying more for a better order. The genuine, measurable question is **how much better** each reranker's order is than free retrieval order — and whether the best one clears a cost margin. top-k and the pool size are settled/bounded, not swept as competing arms.

### How retrieval and reranking relate (clearing up a common mix-up)

Retrieval and reranking are often blurred together ("the reranker retrieves the best ones"). The clean mental model is **two stages, one pipeline**:

```
Candidate flow
Query ─▶ RETRIEVAL (05) ─▶ a POOL of candidates ─▶ RERANKING (06) ─▶ ordered top-k ─▶ context (07)
          ├─ searches the whole corpus              ├─ never touches the corpus
          ├─ optimizes RECALL (don't miss it)       ├─ optimizes PRECISION@top (put the best first)
          └─ cheap per document                     └─ expensive per candidate, but only on the pool
```

So, to correct the common phrasing:

- **A reranker does not retrieve.** It receives a fixed pool and only changes the *order*. If the answer is not in the pool, reranking cannot help.
- **Recall is set upstream; reranking cannot raise it.** It can only move already-retrieved passages up or down. Recall@k of the *pool* is the ceiling.
- **"Semantic reranking" ≠ "semantic retrieval."** Dense *retrieval* finds nearest vectors in the whole corpus; the bi-encoder *reranker* re-applies that same cosine signal to the small pool — identical math, different scope. A cross-encoder/LLM reranker uses a *different, stronger* signal (query and passage read together).
- **What actually "makes reranking worth it" is a better judge than retrieval used** — which is why the LLM/cross-encoder arm exists, and why we only adopt it when its gain clears the cost margin.

## 3. Run Controlled Experiments

- **Fixed upstream:** the index, the selected retrieval method, the top-k, and the RRF constant are held constant; each question's **candidate pool is retrieved once** and frozen, so every reranker reorders the *same* rows.
- **What the code does:** builds the three rerankers — **none** (identity), **bi-encoder semantic** (re-score the pool by query↔passage cosine), and **bounded listwise LLM** (one capped chat call orders the pool) — then embeds the labeled questions and replays the stage-05 method to produce one fixed pool per question.
- **Query embedding & LLM calls:** questions are embedded with the same keyless Azure deployment and on-disk cache as notebooks 03–05 (usually **0** calls — already cached); the LLM reranker caches each ordering on disk keyed by the query and pool, so re-runs cost nothing.
- **Bounded by design:** the LLM sees only the pool, each passage truncated to a character cap, in a single call per query. On a cache miss with Azure unreachable it raises (no fabricated order), consistent with notebook 05.
- **Expected result:** the pool size, the three registered rerankers, the number of questions embedded (and any live embedding calls), and the average candidate pool size produced by the selected method.

In [ ]:
import hashlib
import time

from dotenv import dotenv_values
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from openai import AzureOpenAI

RERANK_POOL = max(TOP_K * 4, 20)   # candidates retrieved once per query, then reordered by each reranker
LLM_SNIPPET_CHARS = 320            # bounded: how much of each passage the LLM sees (caps prompt cost)

# ---- Azure clients (keyless, Entra identity); same on-disk caches as notebooks 03-05 ----
config = {key: value for key, value in dotenv_values(ROOT / '.env').items() if value}
token_provider = get_bearer_token_provider(DefaultAzureCredential(), 'https://cognitiveservices.azure.com/.default')
client = AzureOpenAI(
    azure_endpoint=config['AZURE_OPENAI_ENDPOINT'],
    api_version=config['AZURE_OPENAI_API_VERSION'],
    azure_ad_token_provider=token_provider,
)
embedding_deployment = config['AZURE_OPENAI_EMBEDDING_DEPLOYMENT']
chat_deployment = config['AZURE_OPENAI_CHAT_DEPLOYMENT']

EMBED_CACHE = ARTIFACTS / 'embedding-cache'
RERANK_CACHE = ARTIFACTS / 'rerank-cache'
EMBED_CACHE.mkdir(parents=True, exist_ok=True)
RERANK_CACHE.mkdir(parents=True, exist_ok=True)

embed_api_calls = 0
llm_api_calls = 0
llm_tokens = 0


def embed_query(text):
    """Embed one query, reusing the 03/04/05 on-disk cache; call Azure only on a cache miss.

    The vector is truncated + L2-normalized to the stored dimension so it lives in the indexed vectors' space.
    """
    global embed_api_calls
    cache_file = EMBED_CACHE / (hashlib.sha256(f'{embedding_deployment}|{text}'.encode('utf-8')).hexdigest() + '.json')
    if cache_file.is_file():
        full_vector = json.loads(cache_file.read_text())
    else:
        full_vector = client.embeddings.create(model=embedding_deployment, input=[text]).data[0].embedding
        cache_file.write_text(json.dumps(full_vector))
        embed_api_calls += 1
    return l2_normalize(full_vector[:stored_dims])


# ---- the three reranking arms: each reorders a FIXED candidate pool; none add or remove rows ----
def rerank_none(query_text, query_vector, candidate_rows):
    """Control: keep retrieval's own order untouched — the baseline every reranker must beat."""
    return list(candidate_rows)


def rerank_semantic(query_text, query_vector, candidate_rows):
    """Bi-encoder semantic reranker: re-score the pool by query<->passage cosine, highest first.

    A transparent, offline stand-in for a cross-encoder. Note: when the pool was retrieved by the dense
    method this reproduces that order (same signal) — which is exactly why we also test a stronger LLM arm.
    """
    return sorted(candidate_rows, key=lambda row: dot(query_vector, vectors[row]), reverse=True)


RERANK_SYSTEM = (
    'You are a retrieval reranker for a Shakespeare evidence desk. '
    'Given a question and a numbered list of candidate passages, order the candidates '
    'from most to least relevant to answering the question. Judge only the text shown; '
    'do not use outside knowledge and do not invent passages. '
    'Return a JSON object {"ranking": [numbers]} listing every candidate number exactly once, best first.'
)


def rerank_llm(query_text, query_vector, candidate_rows):
    """Bounded listwise LLM reranker: one capped chat call orders the fixed pool; result cached on disk.

    Bounded = only the pool is sent, each passage truncated to LLM_SNIPPET_CHARS, one call per query.
    On a cache miss with Azure unreachable this raises (no fabricated order), consistent with notebook 05.
    """
    global llm_api_calls, llm_tokens
    pool = list(candidate_rows)
    cache_key = hashlib.sha256(
        f"{chat_deployment}|{LLM_SNIPPET_CHARS}|{query_text}|{','.join(map(str, pool))}".encode('utf-8')).hexdigest()
    cache_file = RERANK_CACHE / (cache_key + '.json')
    if cache_file.is_file():
        order = json.loads(cache_file.read_text())
    else:
        numbered = '\n'.join(
            f"[{position}] ({records[row]['work']}) " + ' '.join(records[row]['text'].split())[:LLM_SNIPPET_CHARS]
            for position, row in enumerate(pool))
        user_prompt = f'Question: {query_text}\n\nCandidates:\n{numbered}'
        kwargs = dict(
            model=chat_deployment,
            messages=[{'role': 'system', 'content': RERANK_SYSTEM},
                      {'role': 'user', 'content': user_prompt}],
        )
        try:
            response = client.chat.completions.create(response_format={'type': 'json_object'}, **kwargs)
        except Exception:
            # Fallback for deployments that reject response_format.
            response = client.chat.completions.create(**kwargs)
        llm_api_calls += 1
        if response.usage:
            llm_tokens += response.usage.total_tokens
        raw = response.choices[0].message.content or '{}'
        match = re.search(r'\{.*\}', raw, re.DOTALL)
        parsed = json.loads(match.group(0) if match else '{}')
        order = [int(n) for n in parsed.get('ranking', []) if isinstance(n, (int, float))]
        cache_file.write_text(json.dumps(order))
    # Map the model's positions back to rows; keep valid positions once, then append any it dropped.
    seen = set()
    reranked = []
    for position in order:
        if 0 <= position < len(pool) and position not in seen:
            seen.add(position)
            reranked.append(pool[position])
    reranked.extend(pool[position] for position in range(len(pool)) if position not in seen)
    return reranked


RERANKERS = {'none': rerank_none, 'semantic': rerank_semantic, 'llm-listwise': rerank_llm}

# Embed the answer-key questions once (cached), then freeze one candidate pool per question.
query_vectors = [embed_query(question['question']) for question, _ in answer_key]
candidate_pools = [retrieve_pool(question['question'], query_vector, RERANK_POOL)
                   for (question, _), query_vector in zip(answer_key, query_vectors)]

avg_pool = sum(len(pool) for pool in candidate_pools) / max(len(candidate_pools), 1)
print(f'Reranking a fixed pool of up to {RERANK_POOL} candidates down to top-{TOP_K} per question.')
print(f'Registered rerankers: {", ".join(RERANKERS)}.')
print(f'Embedded {len(query_vectors)} question(s); {embed_api_calls} embedding call(s) (rest from cache).')
print(f'Average candidate pool size: {avg_pool:.1f} rows (method={SELECTED_METHOD}).')

## 4. Measure and Inspect

- **The lenses:** ranking quality at the top — **nDCG@k** and **Recall@1** — plus **evidence coverage@k**, **MRR**, **p95 latency**, and the **LLM cost** (calls + tokens).
- **nDCG@k — the reranking metric.** Normalized Discounted Cumulative Gain rewards putting relevant passages *high*: each relevant hit contributes less the lower it sits (a `1/log2(rank+1)` discount), normalized by the ideal ordering so it lands in `[0, 1]`. Unlike plain Recall@k (did a relevant row make the top-k at all), nDCG is sensitive to *order* — exactly what reranking changes — so it is the primary measure here.
- **Evidence coverage@k:** the fraction of a question's relevant rows that land in the final top-k. A comparison question may have evidence in several passages; coverage rewards surfacing more of them, which directly feeds context building (07) and faithful citation (08).
- **Recall@1 and MRR:** Recall@1 is how often a relevant passage is ranked *first* (the position the generator weights most); MRR rewards a high first-relevant rank. Reranking should move these up relative to no reranking.
- **p95 latency and LLM cost:** the 95th-percentile per-query rerank time, and the chat calls + tokens the LLM arm spends. None and semantic are near-instant and free; the LLM arm is the slow, paid option — reported honestly so Section 5 can weigh its gain against its cost.
- **A worked reorder:** for one question we print retrieval's top-k beside the LLM's top-k so the reordering is visible by eye, with relevant rows marked.
- **Honest limit:** the labeled dev set is small, so single numbers are coarse. This pre-screens rerankers; the end-to-end judges are generation (08) and continuous evaluation (10).

In [ ]:
K = TOP_K   # grade at the served top-k after reranking


def dcg(gains):
    # Discounted cumulative gain: each gain is discounted by its position (later positions count less).
    return sum(gain / math.log2(position + 2) for position, gain in enumerate(gains))


def grade(reranked_rows, relevant_rows):
    """Score one reranked list's top-K against the relevant rows.

    Returns nDCG@K (binary gains), evidence coverage@K (share of relevant rows surfaced),
    hit@1, hit@K, and the reciprocal rank of the first relevant row.
    """
    top = reranked_rows[:K]
    gains = [1.0 if row in relevant_rows else 0.0 for row in top]
    ideal = [1.0] * min(len(relevant_rows), K)          # best possible: all relevant rows first
    ndcg = dcg(gains) / dcg(ideal) if dcg(ideal) else 0.0
    coverage = sum(1 for row in top if row in relevant_rows) / max(len(relevant_rows), 1)
    first_relevant = next((position for position, row in enumerate(top) if row in relevant_rows), None)
    hit1 = int(first_relevant == 0)
    hitk = int(first_relevant is not None)
    reciprocal_rank = 1.0 / (first_relevant + 1) if first_relevant is not None else 0.0
    return ndcg, coverage, hit1, hitk, reciprocal_rank


def evaluate_reranker(reranker):
    """Run one reranker over every answer-key pool and summarize ranking quality, latency, and cost."""
    ndcg_total = coverage_total = reciprocal_rank_total = 0.0
    hit1_total = hitk_total = 0
    latencies_ms = []
    for (question, relevant_rows), query_vector, pool in zip(answer_key, query_vectors, candidate_pools):
        started = time.perf_counter()
        reranked = reranker(question['question'], query_vector, pool)
        latencies_ms.append((time.perf_counter() - started) * 1000)
        ndcg, coverage, hit1, hitk, reciprocal_rank = grade(reranked, relevant_rows)
        ndcg_total += ndcg
        coverage_total += coverage
        reciprocal_rank_total += reciprocal_rank
        hit1_total += hit1
        hitk_total += hitk
    question_count = max(len(answer_key), 1)
    sorted_latencies = sorted(latencies_ms)
    p95_latency = sorted_latencies[min(len(sorted_latencies) - 1, int(0.95 * len(sorted_latencies)))] if sorted_latencies else 0.0
    return {
        f'ndcg@{K}': ndcg_total / question_count,
        f'coverage@{K}': coverage_total / question_count,
        'recall@1': hit1_total / question_count,
        f'recall@{K}': hitk_total / question_count,
        'mrr': reciprocal_rank_total / question_count,
        'p95_latency_ms': p95_latency,
    }


rerank_metrics = {name: evaluate_reranker(reranker) for name, reranker in RERANKERS.items()}

print(f'Measured over {len(answer_key)} resolved dev question(s); graded at top-{K}.\n')
header = ('reranker', f'ndcg@{K}', f'cover@{K}', 'recall@1', 'mrr', 'p95 ms')
print('{:<14} {:>8} {:>9} {:>9} {:>7} {:>8}'.format(*header))
for name, m in rerank_metrics.items():
    print('{:<14} {:>8.3f} {:>8.0%} {:>8.0%} {:>7.3f} {:>8.2f}'.format(
        name, m[f'ndcg@{K}'], m[f'coverage@{K}'], m['recall@1'], m['mrr'], m['p95_latency_ms']))

print(f'\nLLM reranking cost: {llm_api_calls} Azure chat call(s), {llm_tokens:,} token(s) '
      '(served from cache on re-run).')

# Inspect one question: how the LLM reorders the fixed pool versus retrieval's own order.
if answer_key:
    (first_question, first_relevant) = answer_key[0]
    retrieval_top = candidate_pools[0][:K]
    llm_top = rerank_llm(first_question['question'], query_vectors[0], candidate_pools[0])[:K]

    def label_row(row):
        marker = 'R' if row in first_relevant else '-'   # R = a relevant (cited-scene) row
        return f"[{marker}] {records[row]['work']} {records[row]['scenes']}"

    print(f'\nSample reorder for "{first_question["question"][:66]}..."')
    print('  retrieval top-K:', ' | '.join(label_row(row) for row in retrieval_top))
    print('  LLM top-K:      ', ' | '.join(label_row(row) for row in llm_top))

## 5. Justify the Selection

- **Decision rule (data-driven, deterministic):** pick the reranker with the best **nDCG@k**, but **only adopt a reranker over "none" if its nDCG@k beats no-reranking by at least a margin** (here `0.02`) — reranking is optional and must *earn* its latency and cost. Among arms that clear the margin, prefer **lower p95 latency**, then **lower cost** (LLM calls), then the **simpler** arm. The rule is applied to the measured numbers, not asserted.
- **Why the margin gates adoption:** a reranker that merely ties free retrieval order is pure overhead — extra latency (and, for the LLM, real tokens) for no quality. The margin encodes "don't pay unless the order is genuinely better," which is the whole economics of reranking.
- **Honest, data-dependent outcome (no pre-ordained winner):** on this small, clean pool retrieval may already rank the cited passage at or near the top, leaving little for a reranker to fix — in which case the rule selects **none**. When the pool came from dense retrieval, the bi-encoder arm *reproduces* that order and so cannot clear the margin by construction; only the LLM's stronger judgment can move it. We report whichever the numbers choose rather than assuming the LLM wins.
- **The LLM/cross-encoder stays the production-robust option — and is the production target.** Its value grows as pools get noisier and queries harder, where a cross-encoder reliably out-orders a bi-encoder; on a tiny labeled set that edge may not yet clear the margin. The handoff records the measured local winner *and* names Azure AI Search's **semantic ranker** (a hosted cross-encoder) as the target to re-confirm end-to-end (notebooks 08, 10) — exactly as stages 04/05 selected a local method while naming the managed service in production.
- **Expected result:** the selected reranker with the numbers that justify it (nDCG@k, its gain over no-reranking, MRR, p95 latency), and the alternatives noted with their measured reason.

In [ ]:
MIN_GAIN = 0.02   # minimum nDCG@k improvement over no-reranking to justify a reranker's latency and cost
SIMPLICITY_ORDER = {'none': 0, 'semantic': 1, 'llm-listwise': 2}   # tie-break; lower = simpler

baseline_ndcg = rerank_metrics['none'][f'ndcg@{K}']   # the order every reranker must beat to be adopted


def quality_bucket(name):
    # Bucket the GAIN over no-reranking into MIN_GAIN steps: an improvement below the margin stays in
    # bucket 0 (a tie with 'none'), so only a real gain can outrank the free baseline.
    gain = rerank_metrics[name][f'ndcg@{K}'] - baseline_ndcg
    return math.floor(gain / MIN_GAIN + 1e-9)


def selection_key(name):
    m = rerank_metrics[name]
    # Highest quality bucket first; within a bucket prefer lower latency, then lower cost, then simpler.
    # 'none' is always bucket 0 at zero latency/cost, so it wins unless a reranker clears the margin.
    cost = llm_api_calls if name == 'llm-listwise' else 0
    return (-quality_bucket(name), m['p95_latency_ms'], cost, SIMPLICITY_ORDER.get(name, 99))


selected_reranker = min(rerank_metrics, key=selection_key)
selected = rerank_metrics[selected_reranker]
baseline = rerank_metrics['none']
gain = selected[f'ndcg@{K}'] - baseline[f'ndcg@{K}']

print(f'Selected reranker: {selected_reranker} (pool={RERANK_POOL}, top-k={TOP_K})')
print(f"  ndcg@{K} / coverage@{K} / mrr: {selected[f'ndcg@{K}']:.3f} / {selected[f'coverage@{K}']:.0%} / {selected['mrr']:.3f}")
print(f"  gain over no-reranking:        {gain:+.3f} ndcg@{K}  (margin to adopt: {MIN_GAIN:+.3f})")
print(f"  p95 latency:                   {selected['p95_latency_ms']:.2f} ms/query")
print('\nAlternatives:')
for name, m in rerank_metrics.items():
    if name == selected_reranker:
        continue
    delta = m[f'ndcg@{K}'] - baseline[f'ndcg@{K}']
    if name == 'none':
        reason = 'baseline — kept because no reranker cleared the margin' if selected_reranker == 'none' \
            else 'baseline — beaten by the selected reranker'
    elif delta < MIN_GAIN:
        reason = f'gain {delta:+.3f} below the {MIN_GAIN:+.3f} margin — not worth its cost'
    else:
        reason = 'cleared the margin but costlier / slower than the selection'
    print(f"  {name}: ndcg@{K}={m[f'ndcg@{K}']:.3f} mrr={m['mrr']:.3f} p95={m['p95_latency_ms']:.2f}ms — {reason}")

## 6. Export the Handoff

- **What gets exported:**
  - `.artifacts/reranking-handoff.json` — the decision record notebook 07 loads: the selected reranker (`none` / `semantic` / `llm-listwise`), the rerank pool size, the served top-k, the adopt-or-not margin, the full candidate comparison (nDCG@k, coverage, recall, MRR, p95 latency), the LLM reranker's bounds and cost, and a back-reference to the stage-05 retrieval decision and the stage-04 index.
- **Why a config handoff, not a data dump:** the index (`index.json`) holds the rows and vectors and the retrieval handoff holds how to build the pool; reranking adds only a *decision* — how to reorder that pool — so the handoff is the reproducible configuration plus the evidence for it. Notebook 07 replays retrieval + the selected reranker to reproduce the exact ordered candidates.
- **Provenance preserved:** nothing about the rows changes here; candidates still carry `chunk_id`, work, scenes, and speakers, so context building and citations downstream resolve back to the source lines.
- **Hygiene:** `.artifacts/` is gitignored and notebook outputs are stripped on commit — do not commit the handoff.
- **Expected result:** the handoff written, with the selected reranker, pool size, top-k, and upstream method printed.

In [ ]:
reranking_handoff_path = ARTIFACTS / 'reranking-handoff.json'

# reranking-handoff.json: the decision record notebook 07 replays — the selected reordering of the fixed
# stage-05 pool, plus the measured comparison that justified it. index.json + retrieval-handoff.json hold the rest.
reranking_handoff = {
    'stage': '06-reranking',
    'selected_reranker': selected_reranker,          # none | semantic | llm-listwise
    'rerank_pool': RERANK_POOL,
    'top_k': TOP_K,
    'min_gain_ndcg': MIN_GAIN,
    'selection_rule': 'best nDCG@k that beats no-reranking by >= margin, then lower latency, then lower cost, then simplicity',
    'retrieval_reference': {                          # the fixed upstream this stage reorders
        'stage': retrieval_handoff.get('stage', '05-retrieval'),
        'selected_method': SELECTED_METHOD,
        'top_k': retrieval_handoff.get('top_k'),
        'index_reference': retrieval_handoff.get('index_reference', {}),
    },
    'llm_reranker': {                                 # the bounds and measured cost of the LLM arm
        'deployment': chat_deployment,
        'style': 'listwise',
        'bounded_pool': RERANK_POOL,
        'snippet_chars': LLM_SNIPPET_CHARS,
        'api_calls': llm_api_calls,
        'tokens': llm_tokens,
    },
    'candidates': rerank_metrics,                     # ndcg, coverage, recall, mrr, p95 for every reranker
    'resolved_questions': len(answer_key),
    'production_target': {'service': 'azure-ai-search', 'feature': 'semantic ranker (L2 cross-encoder reranking)'},
}
reranking_handoff_path.write_text(json.dumps(reranking_handoff, ensure_ascii=False, indent=2), encoding='utf-8')

print(f'Wrote decision record to {reranking_handoff_path.relative_to(ROOT)}')
print(f'Selected: {selected_reranker} | pool={RERANK_POOL} | top-k={TOP_K} | upstream method={SELECTED_METHOD}')
print('Reminder: .artifacts/ is gitignored and notebook outputs are stripped on commit — do not commit the handoff.')

## Observations and Open Questions

Honest notes from working through this stage — what the numbers do and do not prove, and what to revisit later.

- **Reranking can only move passages the pool already contains.** Every metric here is bounded by the pool's recall. If a relevant passage never made the stage-05 pool, no reranker recovers it — the fix lives upstream in retrieval (05) or chunking (02), not here.

- **The baseline is the hardest competitor.** On a small, clean corpus retrieval often ranks the cited scene near the top already, so "none" is strong and a reranker must clear a real margin to beat it. That is by design: the margin stops us from paying latency and tokens for a reorder that does not help.

- **The bi-encoder reranker is only as new as its signal.** When the pool came from dense retrieval, re-scoring by the same cosine reproduces the same order — a genuine tie, not a bug. It earns its keep only on pools from BM25 or hybrid, where it re-injects the semantic axis. A real cross-encoder (query + passage read jointly) is the signal that actually differs; the LLM arm stands in for it locally.

- **The LLM reranker is powerful, expensive, and swayable.** It reads query and passages together and can spot true answerhood a vector misses — but it is the slow, paid arm, and it can reward fluent-but-wrong passages. We bound it (small pool, truncated snippets, one call, disk cache) and adopt it only when its gain clears the margin.

- **nDCG is the right lens for this stage.** Recall@k barely moves under reranking (the same rows are present); what changes is their *order*, which nDCG and Recall@1 capture and plain Recall@k does not. That is why the selection gates on nDCG@k, not Recall@k.

- **Latency differences are small now, decisive later.** At ten plays and a pool of twenty, even the LLM call is tolerable; at production scale and query rate, cross-encoder reranking is where the latency budget is spent, and the managed Azure AI Search semantic ranker is measured on its own terms there.

- **How we will revisit.** Reranking is a swappable layer: change the arm, pool size, or margin, re-export the handoff, and re-run 07→08 on the **same** questions. Iterate on the **development** set; check the **held-out** set only periodically so evaluation stays honest.

## Completion Check and Next Notebook

- **Restart and run in order:** restart the kernel and run top to bottom to confirm this notebook depends only on the notebook 04/05 artifacts, not hidden state. Reranking adds no retrieval; only question embedding and the LLM reranker touch Azure, and both are served from disk cache when available.
- **What we achieved:** three rerankers (none, bi-encoder semantic, bounded listwise LLM) run over a fixed candidate pool and graded on nDCG@k, evidence coverage, Recall@1, MRR, p95 latency, and LLM cost; a worked reorder shown by eye; a deterministic, margin-gated selection that adopts a reranker only when it earns its cost; and the exported reranking handoff.
- **Honest limits:** the labeled dev set is small, so the baseline is strong and the arms can tie; the bounded LLM is a local stand-in for a production cross-encoder (Azure AI Search semantic ranker). The real judges are generation (notebook 08) and continuous evaluation (notebook 10).
- **Privacy and hygiene:** outputs are stripped on commit by `nbstripout`; `.artifacts/` is ignored. Do not commit the handoff.
- **Next stage — 07 Context Building:** load `.artifacts/reranking-handoff.json`, replay retrieval + the selected reranker to produce ordered candidates, then dedupe them, optionally fetch parent context, and pack them into a grounded, cited context within a token budget — the last step before generation.

## Questions, Discussions, and Further Insights

The "Observations and Open Questions" cell records caveats about *our* numbers. These questions build the transferable intuition.

**Q1. If we already have good retrieval, why rerank at all?**
Because retrieval optimizes *recall* over the whole corpus, not *order* within the pool. It is tuned to get a relevant passage in cheaply, even if it lands at rank 8. The generator reads the top few, so order at the top decides what evidence actually gets used. Reranking spends a stronger, more expensive judgment on the small pool to pull the best passage to rank 1. It is precision-at-the-top insurance — worth buying only when the order was imperfect and the reranker's judgment is genuinely better.

**Q2. What is the difference between a bi-encoder and a cross-encoder reranker?**
A **bi-encoder** embeds the query and each passage *separately* and compares the two vectors (cosine) — fast, because passage vectors can be precomputed, but it never lets the query and passage "see" each other. A **cross-encoder** feeds the query and passage *together* into one model that outputs a single relevance score — far more accurate, because it can attend to exactly which words answer the question, but it must run at query time for every candidate, so it is only affordable on a small reranking pool. Dense retrieval is a bi-encoder; the production reranker (and our LLM stand-in) is cross-encoder-style.

**Q3. Why does the bi-encoder reranker sometimes change nothing?**
Because it re-applies the *same* cosine signal the dense retriever already used. If the pool came from dense search, re-scoring by query↔passage cosine reproduces the retrieval order exactly — a tie by construction, not a failure. It only reorders pools built by a *different* signal (BM25, hybrid). To genuinely change a dense pool's order you need a *different* judge — a cross-encoder or LLM.

**Q4. Why listwise LLM reranking instead of scoring each passage?**
Pointwise scoring makes one model call per candidate; listwise makes **one** call that sees the whole numbered pool and returns an ordering. Listwise is far cheaper, and it lets the model compare passages against each other rather than in isolation. Its risk is that a long list can destabilize the ordering or drop items — so we bound the pool, truncate snippets, and defensively re-append anything the model omits.

**Q5. Why adopt a reranker only if it beats "none" by a margin?**
Because reranking is pure overhead unless it improves the order. A reranker that ties free retrieval order still costs latency — and, for the LLM, real tokens — for zero quality gain. The margin (`0.02` nDCG@k here) encodes "don't pay unless the order is genuinely better," which is the economics of the whole stage. It is why the rule can legitimately select **none**.

**Q6. Can reranking rescue a passage retrieval missed?**
No. Reranking only reorders the fixed pool; it cannot pull in a passage that was never retrieved. Its quality is capped by the pool's recall. If the right passage is absent, the fix is upstream — widen top-k, change the retrieval method (05), or rechunk (02) — not a better reranker.

**Q7. What did we produce, and what happens next?**
We exported `.artifacts/reranking-handoff.json` — the selected reranker, the pool size, the served top-k, the adopt margin, the LLM bounds and cost, and the full measured comparison that justified the choice. Notebook 07 (Context Building) replays retrieval + the selected reranker to produce ordered candidates, then dedupes, optionally adds parent context, and packs them into a cited context within a token budget for the generator.